### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so why do parrots talk? Hmm, I remember seeing parrots in movies or TV shows where they mimic human speech. But why do they do that in real life? Let me think. I think it has something to do with their natural behavior. Maybe they use it for communication with other parrots. But humans don\'t have other parrots around, so why would they talk to us? Wait, maybe they imitate sounds they hear in their environment as a way to blend in or for social interaction. \n\nParrots are social animals, right? So in the wild, they might mimic sounds to communicate with their flock. If a parrot is in a human home, they might mimic the humans around them. But why specifically talk? Some birds can mimic other sounds like phones, alarms, or other animals. So talking is just another form of that. But what\'s the evolutionary reason? Maybe it\'s a way to learn and adapt to their environment, which helps in survival. For example, if a parrot can mimic a predator\'s call, it

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. So I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'cwzt9k5k0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 154, 'total_tokens': 241, 'completion_time': 0.13929729, 'completion_tokens_details': {'reasoning_tokens': 63}, 'prompt_time': 0.00741544, 'prompt_tokens_details': None, 'queue_time': 0.047488334, 'total_time': 0.14671273}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e2516-a239-7fd1-8

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny. Let me know if you need more details! ☀️


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'epfq66a3h', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.150685168, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006336565, 'prompt_tokens_details': None, 'queue_time': 0.045595001, 'total_time': 0.157021733}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_